### Implementation with 300 seconds timer

In [22]:
import json
import sqlite3
import time
import logging
from datetime import datetime
from pathlib import Path
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ── LOGGING ────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler("scraper.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# ── PATHS & CONSTANTS ──────────────────────────────────────────────────────────
DB_PATH = Path.cwd() / "Data" / "polymarket_gamma_dynamic.sqlite"
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

NUMERIC_COLUMNS = {
    "volume", "volumeNum", "liquidity", "liquidityNum",
    "lastTradePrice", "bestBid", "bestAsk",
    "spread", "volume24hr", "volume1wk", "volume1mo",
}

# SUGGESTION 6: Key fields to monitor for null-rate drift
KEY_FIELDS_TO_MONITOR    = {"volume", "lastTradePrice", "bestBid", "bestAsk"}
NULL_RATE_ALERT_THRESHOLD = 0.10  # warn if >10% of rows are null on a key field

# ── LIFECYCLE STATE ────────────────────────────────────────────────────────────
# SUGGESTION 5: Consecutive-failure alerting + heartbeat
_consecutive_failures            = 0
_run_count                       = 0
CONSECUTIVE_FAILURE_ALERT_THRESHOLD = 3
HEARTBEAT_EVERY_N_RUNS           = 12  # every ~1 hour at 5-min intervals

# ── HTTP SESSION ───────────────────────────────────────────────────────────────
def get_retrying_session():
    """Creates a requests session that automatically retries failed requests."""
    session = requests.Session()
    session.headers.update({"User-Agent": "PolymarketGoldScraper/1.0"})
    retry = Retry(total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)
    return session

http_session = get_retrying_session()

# ── TAG CACHE ──────────────────────────────────────────────────────────────────
CACHED_TAG_IDS      = []
TAG_CACHE_LAST_FETCH = 0.0
TAG_CACHE_TTL        = 3600  # refresh every hour

# ── DATABASE ───────────────────────────────────────────────────────────────────
def init_db():
    # SUGGESTION 3: timeout=30 — SQLite waits up to 30s for a locked DB
    # instead of immediately raising OperationalError
    conn = sqlite3.connect(DB_PATH, timeout=30)
    conn.execute("PRAGMA journal_mode=WAL;")
    return conn

# ── SCRAPER ────────────────────────────────────────────────────────────────────
def gamma_scraper(endpoint="markets", fetch_all=True, **kwargs):
    """
    SUGGESTION 2: Returns None on failure, [] on genuine empty response.
    Callers can now distinguish a broken request from an API returning no data.
    """
    base_url = f"https://gamma-api.polymarket.com/{endpoint}"
    params = {
        "limit":  kwargs.get("limit",  100),
        "offset": kwargs.get("offset", 0),
        "active": str(kwargs.get("active", "true")).lower(),
        "closed": str(kwargs.get("closed", "false")).lower(),
    }
    params.update(kwargs)

    try:
        if not fetch_all:
            response = http_session.get(base_url, params=params, timeout=(5, 30))
            response.raise_for_status()
            return response.json()

        all_items = []
        limit  = int(params.get("limit",  100))
        offset = int(params.get("offset", 0))

        while True:
            current_params = dict(params)
            current_params["limit"]  = limit
            current_params["offset"] = offset

            response = http_session.get(base_url, params=current_params, timeout=(5, 30))
            response.raise_for_status()
            page_data = response.json()

            if isinstance(page_data, list):
                all_items.extend(page_data)
                if len(page_data) < limit:
                    break
                offset += limit
            else:
                return page_data  # single-object response, pass through as-is

        return all_items  # [] is a valid empty result

    except Exception as e:
        logger.error(f"Error fetching /{endpoint}: {e}")
        return None  # None signals a hard failure to the caller

# ── PIPELINE ───────────────────────────────────────────────────────────────────
def save_pipeline_dynamic(conn, data, table_name="markets"):
    """Saves a batch to SQLite with typed columns, write retry, and quality monitoring."""
    if not data:
        return 0

    # SUGGESTION 4: Millisecond-precision timestamp for better time-series resolution
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
    for item in data:
        item["scraped_at"] = current_time

    cursor = conn.cursor()
    all_keys = set()
    for item in data:
        all_keys.update(item.keys())

    columns = sorted(list(all_keys))
    if "scraped_at" in columns: columns.remove("scraped_at")
    if "id" in columns:         columns.remove("id")
    columns.insert(0, "id")
    columns.insert(1, "scraped_at")

    def _col_type(col):
        return "REAL" if col in NUMERIC_COLUMNS else "TEXT"

    col_defs = [f'"{col}" {_col_type(col)}' for col in columns]
    cursor.execute(f"""
        CREATE TABLE IF NOT EXISTS {table_name} (
            {', '.join(col_defs)},
            PRIMARY KEY ("id", "scraped_at")
        )
    """)

    if table_name == "markets":
        cursor.execute(f'CREATE INDEX IF NOT EXISTS idx_{table_name}_scraped_at ON {table_name}("scraped_at")')
        cursor.execute(f'CREATE INDEX IF NOT EXISTS idx_{table_name}_id ON {table_name}("id")')

    cursor.execute(f"PRAGMA table_info({table_name})")
    existing_columns = {row[1] for row in cursor.fetchall()}
    for col in columns:
        if col not in existing_columns:
            cursor.execute(f'ALTER TABLE {table_name} ADD COLUMN "{col}" {_col_type(col)}')

    # SUGGESTION 6: Count numeric coercion failures
    coerce_failures = 0
    rows_to_insert  = []
    for item in data:
        row = []
        for col in columns:
            val = item.get(col)
            if isinstance(val, (dict, list, bool)):
                row.append(json.dumps(val))
            elif col in NUMERIC_COLUMNS and val is not None:
                try:
                    row.append(float(val))
                except (ValueError, TypeError):
                    row.append(None)
                    coerce_failures += 1
            else:
                row.append(val)
        rows_to_insert.append(tuple(row))

    if coerce_failures > 0:
        logger.warning(f"[{table_name}] {coerce_failures} numeric coercion failure(s) — stored as NULL.")

    # SUGGESTION 6: Null-rate drift monitoring on key fields
    if table_name == "markets":
        n = len(data)
        for field in KEY_FIELDS_TO_MONITOR:
            if field in columns:
                null_count = sum(1 for item in data if item.get(field) is None)
                null_rate  = null_count / n
                if null_rate > NULL_RATE_ALERT_THRESHOLD:
                    logger.warning(
                        f"[{table_name}] High null rate on '{field}': "
                        f"{null_rate:.0%} ({null_count}/{n} rows)"
                    )

    insert_sql = (
        f'INSERT OR IGNORE INTO {table_name} '
        f'({", ".join(f"{chr(34)}{c}{chr(34)}" for c in columns)}) '
        f'VALUES ({", ".join(["?"] * len(columns))})'
    )

    # SUGGESTION 3: SQLite write retry on OperationalError (e.g. brief DB lock)
    for attempt in range(3):
        try:
            cursor.executemany(insert_sql, rows_to_insert)
            conn.commit()
            break
        except sqlite3.OperationalError as e:
            if attempt < 2:
                wait = 2 ** attempt
                logger.warning(f"[{table_name}] DB write error (attempt {attempt+1}/3): {e}. Retrying in {wait}s...")
                time.sleep(wait)
            else:
                logger.error(f"[{table_name}] DB write failed after 3 attempts.", exc_info=True)
                raise

    # SUGGESTION 6: Log ignored (duplicate) rows
    rows_inserted = cursor.rowcount
    rows_ignored  = len(rows_to_insert) - rows_inserted
    if rows_ignored > 0:
        logger.info(f"[{table_name}] {rows_inserted} new rows inserted, {rows_ignored} duplicate rows ignored.")

    return rows_inserted

# ── JOB ────────────────────────────────────────────────────────────────────────
def run_job(keyword="gold", limit_results=None):
    """Single scraping run with lifecycle monitoring, quality checks, and failure alerting."""
    global CACHED_TAG_IDS, TAG_CACHE_LAST_FETCH, _consecutive_failures, _run_count

    _run_count += 1
    limit_text = "All" if limit_results is None else limit_results
    logger.info(f"[Run #{_run_count}] Starting scrape (keyword='{keyword}', limit={limit_text})...")

    # SUGGESTION 5: Heartbeat — proves the process is alive in the log file
    if _run_count % HEARTBEAT_EVERY_N_RUNS == 0:
        logger.info(
            f"[HEARTBEAT] Run #{_run_count} — scraper alive. "
            f"Consecutive failures: {_consecutive_failures}."
        )

    try:
        conn = init_db()

        # Tag cache with TTL
        cache_age = time.time() - TAG_CACHE_LAST_FETCH
        if not CACHED_TAG_IDS or cache_age > TAG_CACHE_TTL:
            logger.info(f"Tag cache stale ({int(cache_age)}s old). Fetching from API...")
            all_tags = gamma_scraper(endpoint="tags", limit=100)
            # SUGGESTION 2: None = hard failure, abort this run cleanly
            if all_tags is None:
                raise RuntimeError("Tag fetch returned None (request failed). Cannot proceed.")
            target_keywords = ["finance", "crypto", "geopolitics", "politics"]
            CACHED_TAG_IDS = list({
                tag.get("id")
                for tag in all_tags
                if any(
                    tk in str(tag.get("label", "")).lower() or
                    tk in str(tag.get("slug",  "")).lower()
                    for tk in target_keywords
                )
            })
            TAG_CACHE_LAST_FETCH = time.time()
            save_pipeline_dynamic(conn, all_tags, "tags")
            logger.info(f"Cached {len(CACHED_TAG_IDS)} tag IDs for the next {TAG_CACHE_TTL}s.")
        else:
            logger.info(f"Using {len(CACHED_TAG_IDS)} cached tag IDs ({int(cache_age)}s old).")

        # SUGGESTION 2: Per-tag success/failure tracking
        raw_markets      = {}
        tag_success_count = 0
        tag_failure_count = 0

        for t_id in CACHED_TAG_IDS:
            tag_markets = gamma_scraper(endpoint="markets", tag_id=t_id, limit=100)
            if tag_markets is None:      # hard failure — request broke
                tag_failure_count += 1
            else:                        # [] or populated list — both are valid
                tag_success_count += 1
                for m in tag_markets:
                    raw_markets[m["id"]] = m

        if tag_failure_count > 0:
            logger.warning(
                f"Per-tag fetch summary: {tag_success_count} succeeded, "
                f"{tag_failure_count} FAILED (markets from failed tags excluded)."
            )
        else:
            logger.info(f"Per-tag fetch: all {tag_success_count} tag requests succeeded.")

        market_list = list(raw_markets.values())
        logger.info(f"Found {len(market_list)} unique active markets across targeted tags.")

        # Keyword filter
        if keyword:
            kw = keyword.lower()
            filtered = [
                m for m in market_list
                if kw in str(m.get("question",    "")).lower()
                or kw in str(m.get("description", "")).lower()
            ]
        else:
            filtered = market_list

        filtered.sort(key=lambda x: float(x.get("volume", 0) or 0), reverse=True)
        if limit_results is not None:
            filtered = filtered[:limit_results]

        rows_saved = save_pipeline_dynamic(conn, filtered, "markets")
        logger.info(f"[Run #{_run_count}] Complete — {rows_saved} new rows written to DB.")

        # SUGGESTION 5: Reset consecutive failure counter on clean success
        _consecutive_failures = 0

    except Exception as e:
        _consecutive_failures += 1
        logger.error(f"[Run #{_run_count}] CRITICAL ERROR: {e}", exc_info=True)

        # SUGGESTION 5: Escalate after N consecutive failures
        if _consecutive_failures >= CONSECUTIVE_FAILURE_ALERT_THRESHOLD:
            logger.critical(
                f"ALERT: {_consecutive_failures} consecutive failures. "
                f"Scraper may be stalled — check scraper.log immediately."
            )

    finally:
        if 'conn' in locals():
            conn.close()

In [ ]:
"""Infinite wall-clock loop with graceful shutdown support."""
import signal

# SUGGESTION 5: Graceful shutdown — SIGTERM sets a flag instead of killing mid-run
_shutdown_requested = False

def _request_shutdown(signum, frame):
    global _shutdown_requested
    logger.info("Shutdown signal received. Will stop cleanly after current run completes.")
    _shutdown_requested = True

# SIGTERM is the clean shutdown signal (e.g. from kill <pid> or cloud schedulers)
# In Jupyter, Ctrl+C raises KeyboardInterrupt instead — handled below
signal.signal(signal.SIGTERM, _request_shutdown)

if __name__ == "__main__":
    INTERVAL_SECONDS = 300

    logger.info(f"Initializing Wall-Clock Scraper (interval={INTERVAL_SECONDS}s).")
    logger.info("Send SIGTERM or press Ctrl+C to stop gracefully after current run.")

    try:
        while not _shutdown_requested:
            now          = time.time()
            time_to_wait = INTERVAL_SECONDS - (now % INTERVAL_SECONDS)
            next_run     = datetime.fromtimestamp(now + time_to_wait).strftime('%H:%M:%S')
            logger.info(f"Sleeping {int(time_to_wait)}s until next run at {next_run}...")
            time.sleep(time_to_wait)

            if not _shutdown_requested:
                run_job(keyword="gold", limit_results=None)

    except KeyboardInterrupt:
        logger.info("KeyboardInterrupt received. Scraper stopped cleanly.")

    logger.info(f"Scraper shut down after {_run_count} total run(s).")

2026-03-29 16:41:00,610 [INFO] Initializing Wall-Clock Scraper (interval=300s).
2026-03-29 16:41:00,612 [INFO] Send SIGTERM or press Ctrl+C to stop gracefully after current run.
2026-03-29 16:41:00,613 [INFO] Sleeping 239s until next run at 16:45:00...
2026-03-29 16:45:00,012 [INFO] [Run #1] Starting scrape (keyword='gold', limit=All)...
2026-03-29 16:45:00,046 [INFO] Tag cache stale (1774795500s old). Fetching from API...
2026-03-29 16:45:05,869 [INFO] Cached 34 tag IDs for the next 3600s.
2026-03-29 16:45:44,036 [INFO] Per-tag fetch: all 34 tag requests succeeded.
2026-03-29 16:45:44,037 [INFO] Found 13284 unique active markets across targeted tags.
2026-03-29 16:45:44,191 [WARNING] [markets] High null rate on 'lastTradePrice': 21% (27/129 rows)
2026-03-29 16:45:44,191 [WARNING] [markets] High null rate on 'volume': 21% (27/129 rows)
2026-03-29 16:45:44,207 [INFO] [Run #1] Complete — 129 new rows written to DB.
2026-03-29 16:45:44,313 [INFO] Sleeping 255s until next run at 16:50:00..

In [2]:
'''test single scraper run'''
run_job(keyword="gold", limit_results=10)

NameError: name 'run_job' is not defined

In [3]:
'''test print of market title, volume and scraping time'''


import sqlite3
import json
from pathlib import Path

DB_PATH = Path.cwd() / "Data" / "polymarket_gamma_dynamic.sqlite"
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Removed 'tags' from the SELECT statement
cursor.execute("SELECT question, volume, scraped_at FROM markets LIMIT 100;")
results = cursor.fetchall()

print(f"Found {len(results)} markets. Here is the breakdown:\n")
print("-" * 50)

for row in results:
    question = row[0]
    # Format volume to look like a readable dollar amount
    volume = f"${float(row[1]):,.2f}" if row[1] else "$0.00"
    scraped_at = row[2]

    print(f"Market: {question}")
    print(f"Volume: {volume}")
    print(f"Scraped At: {scraped_at}")
    print("-" * 50)

conn.close()

Found 100 markets. Here is the breakdown:

--------------------------------------------------
Market: Will Gold (GC) hit (HIGH) $5,500 by end of June?
Volume: $788,546.05
Scraped At: 2026-03-29 16:45:44.150
--------------------------------------------------
Market: Will Gold (GC) hit (LOW) $3,000 by end of March?
Volume: $490,248.81
Scraped At: 2026-03-29 16:45:44.150
--------------------------------------------------
Market: Will Bitcoin outperform Gold in 2026?
Volume: $376,976.33
Scraped At: 2026-03-29 16:45:44.150
--------------------------------------------------
Market: Will Bitcoin have the best performance in 2026?
Volume: $370,990.59
Scraped At: 2026-03-29 16:45:44.150
--------------------------------------------------
Market: Will Gold (GC) hit (HIGH) $7,000 by end of March?
Volume: $365,473.67
Scraped At: 2026-03-29 16:45:44.150
--------------------------------------------------
Market: Will Gold (GC) hit (HIGH) $10,000 by end of March?
Volume: $324,555.21
Scraped At: 2026-0

In [5]:
import sqlite3
from pathlib import Path

DB_PATH = Path.cwd() / "Data" / "polymarket_gamma_dynamic.sqlite"
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("""
SELECT question, COUNT(DISTINCT scraped_at) AS snapshots
FROM markets
GROUP BY question
ORDER BY snapshots DESC, question
LIMIT 20;
""")

for question, snapshots in cursor.fetchall():
    print(f"{question} -> {snapshots} unique timestamps")

conn.close()

Gold (XAUUSD) Up or Down on March 30? -> 2 unique timestamps
Trump's face on US gold coin by July 4? -> 2 unique timestamps
Will Bitcoin have the best performance in 2026? -> 2 unique timestamps
Will Bitcoin outperform Gold in 2026? -> 2 unique timestamps
Will Bitcoin outperform Gold in April 2026? -> 2 unique timestamps
Will Bitcoin outperform Gold in March 2026? -> 2 unique timestamps
Will Dan Goldman be the Democratic nominee for NY-10? -> 2 unique timestamps
Will Gold (GC) hit (HIGH) $10,000 by end of December? -> 2 unique timestamps
Will Gold (GC) hit (HIGH) $10,000 by end of June? -> 2 unique timestamps
Will Gold (GC) hit (HIGH) $10,000 by end of March? -> 2 unique timestamps
Will Gold (GC) hit (HIGH) $12,000 by end of December? -> 2 unique timestamps
Will Gold (GC) hit (HIGH) $15,000 by end of December? -> 2 unique timestamps
Will Gold (GC) hit (HIGH) $5,400 by end of March? -> 2 unique timestamps
Will Gold (GC) hit (HIGH) $5,500 by end of June? -> 2 unique timestamps
Will Gold 